In [1]:
!nvidia-smi

Sat Mar 28 02:53:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Update the package list and install Nsight Compute
!apt-get update -yq
!apt-get install -yq nsight-compute

# 2. Add the CUDA binaries folder to your Python environment's PATH
import os
os.environ['PATH'] += ':/usr/local/cuda/bin'

# 3. Verify the installation
!ncu --version

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Pac

In [3]:
!pip install torch torchvision torchsummary

## ResNet-50

### Training

In [4]:
!ncu --profile-from-start off \
--target-processes all \
-o resnet50_train_l4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py --dummy -a resnet50 --epochs 1 --profile-step 10 -b 8

!ncu -i resnet50_train_l4.ncu-rep --csv > resnet50_train_l4.csv

==PROF== Connected to process 5313 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'resnet50'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8      

### Evaluation

In [5]:
!ncu --profile-from-start off \
--target-processes all \
-o resnet50_eval_l4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py -e --dummy -a resnet50 --epochs 1 --profile-step 10 -b 8

!ncu -i resnet50_eval_l4.ncu-rep --csv > resnet50_eval_l4.csv

==PROF== Connected to process 8311 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'resnet50'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8      

## VGG-16

### Training

In [6]:
!ncu --profile-from-start off \
--target-processes all \
-o vgg16_train_l4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py --dummy -a vgg16 --epochs 1 --profile-step 10 -b 8

!ncu -i vgg16_train_l4.ncu-rep --csv > vgg16_train_l4.csv

==PROF== Connected to process 9336 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vgg16'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        [

### Evaluation

In [7]:
!ncu --profile-from-start off \
--target-processes all \
-o vgg16_eval_l4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py -e --dummy -a vgg16 --epochs 1 --profile-step 10 -b 8

!ncu -i vgg16_eval_l4.ncu-rep --csv > vgg16_eval_l4.csv

==PROF== Connected to process 10641 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vgg16'
Input size: (3, 224, 224)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        

## ViT-B-16

### Training

In [8]:
!ncu --profile-from-start off \
--target-processes all \
-o vit_train_l4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py --dummy -a vit_b_16 --epochs 1 --profile-step 10 -b 8

!ncu -i vit_train_l4.ncu-rep --csv > vit_train_l4.csv

==PROF== Connected to process 11104 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vit_b_16'
Input size: (3, 224, 224)
=> Skipping torchsummary for vit_b_16 to prevent hook corruption.
=> Dummy data is used!
Epoch: [0][     1/160146]	Time  1.412 ( 1.412)	Data  0.192 ( 0.192)	Loss 6.9078e+00 (6.9078e+00)	Acc@1   0.00 (  0.00)	Acc@5   0.00 (  0.00)
==PROF== Profiling "computeOffsetsKernel" - 0: 0%....50%....100% - 3 passes
==PROF== Profiling "_5x_cudnn_ampere_scudnn_128x6..." - 1: 0%....50%....100% - 3 passes
==PROF== Profiling "elementwise_kernel" - 2: 0%....50%....100% - 3 passes
==PROF== Profiling "CatArrayBatchedCopy" - 3: 0%....50%....100% - 3 passes
==PROF== Profiling "elementwise_kernel" - 4: 0%....50%....1

### Evaluation

In [9]:
!ncu --profile-from-start off \
--target-processes all \
-o vit_eval_l4 -f \
--metrics smsp__sass_thread_inst_executed_op_fadd_pred_on.sum,smsp__sass_thread_inst_executed_op_fmul_pred_on.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__sass_thread_inst_executed_op_hmma_pred_on.sum,dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum \
python main.py -e --dummy -a vit_b_16 --epochs 1 --profile-step 10 -b 8

!ncu -i vit_eval_l4.ncu-rep --csv > vit_eval_l4.csv

==PROF== Connected to process 14417 (/usr/bin/python3.12)
Using device: cuda
/content/main.py:125: UserWarning: nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'
  warnings.warn("nccl backend >=2.5 requires GPU count>1, see https://github.com/NVIDIA/nccl/issues/103 perhaps use 'gloo'")
=> creating model 'vit_b_16'
Input size: (3, 224, 224)
=> Skipping torchsummary for vit_b_16 to prevent hook corruption.
=> Dummy data is used!
Test: [   1/6250]	Time  1.017 ( 1.017)	Loss 6.9078e+00 (6.9078e+00)	Acc@1   0.00 (  0.00)	Acc@5   0.00 (  0.00)
==PROF== Profiling "computeOffsetsKernel" - 0: 0%....50%....100% - 3 passes
==PROF== Profiling "_5x_cudnn_ampere_scudnn_128x6..." - 1: 0%....50%....100% - 3 passes
==PROF== Profiling "elementwise_kernel" - 2: 0%....50%....100% - 3 passes
==PROF== Profiling "CatArrayBatchedCopy" - 3: 0%....50%....100% - 3 passes
==PROF== Profiling "elementwise_kernel" - 4: 0%....50%....100% - 3 passes
==PROF== Profi